# Smart City NYC — Urban Data Processing using Spark DataFrames

**Case Study 2 · Unit 5.2 DataFrames · Tools: PySpark, Pandas**

| Name | Roll No |
|---|---|
| Varun Murugan | 124A8118 |
| Rankin Yanu | 124A8124 |
| Navin Nadar | 225A8134 |
| Abhay Hanchate | 225A8129 |

**Idea:** a smart city collects data from many sources: vehicles, citizens, sensors and road networks.
We load five open New York City data sources into Spark DataFrames, clean them, combine them, and answer
questions a city operations team would ask. Then we optimize the processing and measure the speed-ups.

**Data sources (Parquet + CSV + JSON + REST APIs, one of them live):**

| Smart city domain | Data | Format / source | Size |
|---|---|---|---|
| 🚕 Mobility | NYC Yellow Taxi trips, Jan 2024 | Parquet (NYC TLC) | ~3M rows |
| 🚕 Mobility | Taxi zone lookup | **CSV** (NYC TLC) | 265 rows |
| 🏛️ Citizen services | 311 service requests (complaints), Jan 2024 | **CSV from the NYC Open Data API** | ~250k rows |
| 🌫️ Environment | Hourly air quality (PM2.5, NO₂, CO), Jan 2024 | **JSON from the Open-Meteo Air Quality API** | 744 rows |
| 🌦️ Environment | Hourly weather (temperature, rain), Jan 2024 | **JSON from the Open-Meteo Weather API** | 744 rows |
| 🚦 Live traffic | Real-time road speeds from city traffic sensors | **Live JSON API** (NYC DOT, refreshed every few minutes) | latest readings |

**How to run:** in Google Colab use *Runtime → Run all*. The notebook installs what it needs and downloads all data itself.
Cells marked 📸 show where to take a screenshot for the report.

**Notebook flow (the 7 required steps):**
1. Setup environment → 2. Load datasets → 3. Distributed processing → 4. Transformations & actions (cleaning + 10 queries)
→ 5. Performance optimization → 6. Capture results → 7. Performance analysis

## Step 1 — Setup environment

We run Spark in **local mode** (`local[*]`): one machine, using every CPU core as a parallel worker.
The same DataFrame code runs unchanged on a real cluster; only the `master` setting changes.

In [ ]:
# Install PySpark / Java only if missing (Colab usually has both)
import importlib.util, os, re, shutil, subprocess, sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)
import pyspark

def java_major():
    if shutil.which("java") is None:
        return 0
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    found = re.search(r'version "(?:1\.)?(\d+)', out)
    return int(found.group(1)) if found else 0

needed = 17 if int(pyspark.__version__.split(".")[0]) >= 4 else 8   # Spark 4 needs Java 17+
if java_major() < needed:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
print(f"PySpark {pyspark.__version__} with Java {java_major()} is ready")

In [ ]:
import os, time, statistics, urllib.request
from functools import reduce

import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, StringType

def create_spark(master="local[*]"):
    return (SparkSession.builder
            .appName("NYC-Taxi-DataFrames-CaseStudy")
            .master(master)
            .config("spark.driver.memory", "4g")
            .config("spark.sql.adaptive.enabled", "true")   # Adaptive Query Execution (default in Spark 3.2+)
            .getOrCreate())

spark = create_spark()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version       :", spark.version)
print("PySpark version     :", pyspark.__version__)
print("Pandas version      :", pd.__version__)
print("Master              :", spark.sparkContext.master)
print("Parallel cores      :", spark.sparkContext.defaultParallelism)
print("Spark UI            :", spark.sparkContext.uiWebUrl)

> 📸 **SCREENSHOT 1** — the output above (Spark version, master, number of cores).

**Open the Spark UI.** In Colab, the cell below opens it in a new window (on a laptop, open http://localhost:4040).
Keep it open: Steps 3–5 ask for screenshots of its tabs.

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(4040, path="/jobs/", anchor_text="Open Spark UI")
except ImportError:
    print("Not in Colab, open:", spark.sparkContext.uiWebUrl)

## Step 2 — Load datasets (Parquet, CSV, JSON from REST APIs, live API)

The cell below downloads every source into `data/raw/` (files that already exist are skipped).
* The **311 complaints** come from the NYC Open Data (Socrata) API in pages of 100,000 rows, saved as CSV files.
* **Air quality** and **weather** come from Open-Meteo REST APIs as JSON.
* **Live traffic** is fetched fresh on every run, so it shows the city's road speeds *right now*.

In [ ]:
import json
from urllib.parse import urlencode

RAW, PROCESSED = "data/raw", "data/processed"
CHARTS, RESULTS = "outputs/charts", "outputs/results"
for d in (RAW, PROCESSED, CHARTS, RESULTS):
    os.makedirs(d, exist_ok=True)

TRIPS_PATH   = f"{RAW}/yellow_tripdata_2024-01.parquet"
ZONES_PATH   = f"{RAW}/taxi_zone_lookup.csv"
WEATHER_PATH = f"{RAW}/weather_nyc_2024-01.json"
AIR_PATH     = f"{RAW}/air_quality_nyc_2024-01.json"
C311_DIR     = f"{RAW}/nyc_311_2024-01"
TRAFFIC_DIR  = f"{RAW}/traffic_live"

OPEN_METEO = "latitude=40.71&longitude=-74.01&start_date=2024-01-01&end_date=2024-01-31&timezone=America/New_York"
FILES = {
    TRIPS_PATH:   "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    ZONES_PATH:   "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
    WEATHER_PATH: f"https://archive-api.open-meteo.com/v1/archive?{OPEN_METEO}&hourly=temperature_2m,precipitation,rain",
    AIR_PATH:     f"https://air-quality-api.open-meteo.com/v1/air-quality?{OPEN_METEO}"
                  "&hourly=pm2_5,nitrogen_dioxide,carbon_monoxide",
}

def http_get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return resp.read()

for path, url in FILES.items():
    if not os.path.exists(path):
        print("Downloading", path)
        with open(path, "wb") as f:
            f.write(http_get(url))

# 311 service requests: NYC Open Data (Socrata) API, paged CSV download
SOCRATA_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv"
PAGE = 100_000
if not os.path.exists(f"{C311_DIR}/_COMPLETE"):
    os.makedirs(C311_DIR, exist_ok=True)
    offset, page_no = 0, 0
    while True:
        query = urlencode({
            "$select": "unique_key,created_date,closed_date,agency,complaint_type,descriptor,"
                       "borough,incident_zip,latitude,longitude,status",
            "$where":  "created_date between '2024-01-01T00:00:00' and '2024-01-31T23:59:59'",
            "$order":  "unique_key",
            "$limit":  PAGE, "$offset": offset})
        body = http_get(f"{SOCRATA_311}?{query}")
        rows = body.count(b"\n") - 1                       # minus the header line
        with open(f"{C311_DIR}/part_{page_no:02d}.csv", "wb") as f:
            f.write(body)
        print(f"311 API page {page_no}: {rows:,} rows")
        if rows < PAGE:
            break
        offset, page_no = offset + PAGE, page_no + 1
    open(f"{C311_DIR}/_COMPLETE", "w").close()

# Live traffic speeds: NYC DOT real-time API, fetched fresh on every run
os.makedirs(TRAFFIC_DIR, exist_ok=True)
try:
    query = urlencode({"$select": "id,speed,travel_time,status,data_as_of,link_id,borough,link_name",
                       "$order": "data_as_of DESC", "$limit": 5000})
    body = http_get(f"https://data.cityofnewyork.us/resource/i4gi-tjb9.json?{query}")
    TRAFFIC_PATH = f"{TRAFFIC_DIR}/traffic_{time.strftime('%Y%m%d_%H%M%S')}.json"
    with open(TRAFFIC_PATH, "wb") as f:
        f.write(body)
    print("Fetched live traffic snapshot ->", TRAFFIC_PATH)
except Exception as err:                                   # offline: reuse the latest snapshot on disk
    saved = sorted(f for f in os.listdir(TRAFFIC_DIR) if f.endswith(".json"))
    if not saved:
        raise RuntimeError("Live traffic API unreachable and no saved snapshot; re-run this cell") from err
    TRAFFIC_PATH = f"{TRAFFIC_DIR}/{saved[-1]}"
    print(f"Live traffic API unreachable ({err.__class__.__name__}); using saved snapshot {TRAFFIC_PATH}")

def size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e6
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6

print()
for path in (TRIPS_PATH, ZONES_PATH, C311_DIR, AIR_PATH, WEATHER_PATH, TRAFFIC_PATH):
    print(f"{path:55s} {size_mb(path):8.2f} MB")

### 2a. 🚕 Taxi trips — Parquet (columnar file, schema stored inside the file)

In [ ]:
trips_raw = spark.read.parquet(TRIPS_PATH)
trips_raw.printSchema()
print(f"Rows: {trips_raw.count():,}   Columns: {len(trips_raw.columns)}")
trips_raw.show(5)

### 2b. 🚕 Taxi zones — CSV with an explicit schema (faster and safer than `inferSchema`)

In [ ]:
zone_schema = StructType([
    StructField("LocationID",   IntegerType()),
    StructField("Borough",      StringType()),
    StructField("Zone",         StringType()),
    StructField("service_zone", StringType()),
])
zones = spark.read.csv(ZONES_PATH, header=True, schema=zone_schema)
print(f"Rows: {zones.count():,}")
zones.show(5, truncate=False)

### 2c. 🏛️ 311 service requests — a folder of CSV files read as **one** DataFrame

Spark reads every file in the folder in parallel. Dates are kept as text here and parsed during cleaning (Step 4).

In [ ]:
c311_schema = StructType([
    StructField("unique_key",     LongType()),
    StructField("created_date",   StringType()),
    StructField("closed_date",    StringType()),
    StructField("agency",         StringType()),
    StructField("complaint_type", StringType()),
    StructField("descriptor",     StringType()),
    StructField("borough",        StringType()),
    StructField("incident_zip",   StringType()),
    StructField("latitude",       DoubleType()),
    StructField("longitude",      DoubleType()),
    StructField("status",         StringType()),
])
c311_raw = spark.read.csv(C311_DIR, header=True, schema=c311_schema)
c311_raw.printSchema()
print(f"Rows: {c311_raw.count():,}   Files: {len(c311_raw.inputFiles())}")
c311_raw.show(5, truncate=30)

### 2d. 🌫️🌦️ Air quality and weather — nested JSON from the APIs

Each API returns one JSON object holding parallel arrays (`time[]`, `pm2_5[]`, …).
We zip the arrays together and `explode` them into one row per hour.

In [ ]:
def hourly_api_json(path, fields):
    """Flatten an Open-Meteo JSON response into one row per hour."""
    return (spark.read.option("multiLine", True).json(path)
                 .select("hourly.*")
                 .select(F.explode(F.arrays_zip("time", *fields)).alias("h"))
                 .select(F.to_timestamp("h.time", "yyyy-MM-dd'T'HH:mm").alias("hour_ts"),
                         *[F.col(f"h.{f}").alias(f) for f in fields]))

spark.read.option("multiLine", True).json(AIR_PATH).printSchema()

air = hourly_api_json(AIR_PATH, ["pm2_5", "nitrogen_dioxide", "carbon_monoxide"])
weather = (hourly_api_json(WEATHER_PATH, ["temperature_2m", "precipitation", "rain"])
           .withColumnRenamed("temperature_2m", "temp_c")
           .withColumnRenamed("precipitation", "precip_mm"))
print(f"Air quality rows: {air.count()}   Weather rows: {weather.count()}")
air.show(5)
weather.show(5)

### 2e. 🚦 Live traffic speeds — JSON array from the real-time API

In [ ]:
traffic_raw = spark.read.option("multiLine", True).json(TRAFFIC_PATH)
traffic_raw.printSchema()
print(f"Sensor readings in this snapshot: {traffic_raw.count():,}")
traffic_raw.orderBy(F.desc("data_as_of")).show(5, truncate=40)

> 📸 **SCREENSHOT 2** — the download sizes, and the schemas and row counts of all datasets (cells above).

## Step 3 — Distributed processing

A DataFrame is split into **partitions**. Each partition is processed by a separate **task**, and tasks run in
parallel on the available cores (or on cluster executors). More partitions → more parallel tasks.

In [ ]:
print("Cores available            :", spark.sparkContext.defaultParallelism)
print("Partitions in trips_raw    :", trips_raw.rdd.getNumPartitions())

print("Rows in each partition:")
trips_raw.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

In [ ]:
# repartition() redistributes the rows evenly into N partitions (this needs a shuffle)
trips_8 = trips_raw.repartition(8)
print("Partitions after repartition(8):", trips_8.rdd.getNumPartitions())
trips_8.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

In [ ]:
# A distributed aggregation: each task builds partial counts for its partition,
# then a shuffle sends rows with the same key to the same task for the final result.
spark.sparkContext.setJobDescription("Step 3: distributed aggregation by vendor")
trips_8.groupBy("VendorID").count().orderBy("VendorID").show()
spark.sparkContext.setJobDescription(None)

> 📸 **SCREENSHOT 3** — the partition counts above.

> 📸 **SCREENSHOT 4** — Spark UI → **Jobs** tab (shows the job for 'Step 3: distributed aggregation by vendor').

> 📸 **SCREENSHOT 5** — Spark UI → **Stages** tab (the aggregation is split into 2 stages because of the shuffle; note Shuffle Read/Write).

> 📸 **SCREENSHOT 6** — Spark UI → **Executors** tab (cores, tasks completed, memory).

## Step 4 — Transformations and actions

| | What it does | Examples |
|---|---|---|
| **Transformation** | Describes a new DataFrame. *Lazy*: nothing runs yet | `select`, `filter`, `withColumn`, `join`, `groupBy`, `orderBy` |
| **Action** | Triggers the actual computation and returns or writes a result | `count`, `show`, `collect`, `toPandas`, `write` |
| *Narrow* transformation | Each output partition needs one input partition → no shuffle | `filter`, `select`, `withColumn` |
| *Wide* transformation | Rows must move between partitions → **shuffle**, new stage | `groupBy`, `join`, `orderBy`, `repartition` |

### 4a. 🚕 Taxi trips: cleaning rules and a data-quality report (computed in **one pass** over the data)

In [ ]:
START, END = F.lit("2024-01-01").cast("timestamp"), F.lit("2024-02-01").cast("timestamp")

prepared = (trips_raw
    .withColumn("pickup_ts",  F.col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
    .withColumn("trip_duration_min",
                (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60)
    .withColumn("speed_mph", F.when(F.col("trip_duration_min") > 0,
                                     F.col("trip_distance") / (F.col("trip_duration_min") / 60)))
    .fillna({"passenger_count": 1}))   # missing passenger count -> assume 1 passenger

quality_rules = {
    "pickup in Jan 2024":   (F.col("pickup_ts") >= START) & (F.col("pickup_ts") < END),
    "duration 1-180 min":   F.col("trip_duration_min").between(1, 180),
    "distance 0-100 miles": (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100),
    "fare and total > 0":   (F.col("fare_amount") > 0) & (F.col("total_amount") > 0),
    "passengers 1-6":       F.col("passenger_count").between(1, 6),
    "speed <= 80 mph":      F.col("speed_mph") <= 80,
}

# Data-quality report: rows FAILING each rule (a row can fail more than one rule)
dq = prepared.agg(
    F.count("*").alias("total rows"),
    *[F.sum(F.when(F.coalesce(rule, F.lit(False)), 0).otherwise(1)).alias(name)
      for name, rule in quality_rules.items()]
).toPandas().T.rename(columns={0: "rows"})
dq

### 4b. Lazy evaluation: defining the cleaned DataFrame is instant; the action does the work

In [ ]:
PAYMENT_NAMES = {0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge",
                 4: "Dispute", 5: "Unknown", 6: "Voided"}
payment_map = F.create_map(*[F.lit(x) for pair in PAYMENT_NAMES.items() for x in pair])
all_rules = reduce(lambda a, b: a & b, quality_rules.values())

t0 = time.perf_counter()
trips = (prepared
    .filter(all_rules)                                                     # narrow
    .withColumn("pickup_hour",    F.hour("pickup_ts"))                     # narrow
    .withColumn("pickup_date",    F.to_date("pickup_ts"))
    .withColumn("day_of_week",    F.date_format("pickup_ts", "E"))
    .withColumn("is_weekend",     F.dayofweek("pickup_ts").isin(1, 7))
    .withColumn("tip_pct",        F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2))
    .withColumn("payment_method", F.coalesce(payment_map[F.col("payment_type")], F.lit("Other")))
    .select("VendorID", "pickup_ts", "dropoff_ts", "pickup_date", "pickup_hour", "day_of_week",
            "is_weekend", "passenger_count", "trip_distance", "trip_duration_min", "speed_mph",
            "PULocationID", "DOLocationID", "payment_method", "fare_amount", "tip_amount",
            "tip_pct", "total_amount"))
t_define = time.perf_counter() - t0

t0 = time.perf_counter()
clean_rows = trips.count()                                                 # ACTION
t_action = time.perf_counter() - t0

raw_rows = int(dq.loc["total rows", "rows"])
print(f"Defining the transformations took {t_define * 1000:8.1f} ms  (lazy: nothing executed)")
print(f"The count() action took          {t_action * 1000:8.1f} ms  (whole pipeline executed)")
print(f"Rows: {raw_rows:,} raw -> {clean_rows:,} clean "
      f"({raw_rows - clean_rows:,} removed, {(raw_rows - clean_rows) / raw_rows:.1%})")
trips.show(5)

> 📸 **SCREENSHOT 7** — the data-quality report and the lazy-evaluation timings above.

### 4c. 🏛️ 311 requests: parse dates, remove duplicates, normalize boroughs, compute resolution time

In [ ]:
c311 = (c311_raw
    .dropDuplicates(["unique_key"])                                         # wide: needs a shuffle
    .withColumn("created_ts", F.expr("try_cast(created_date AS TIMESTAMP)"))
    .withColumn("closed_ts",  F.expr("try_cast(closed_date AS TIMESTAMP)"))
    .filter(F.col("created_ts").isNotNull())
    .withColumn("borough", F.when(F.upper("borough").isin(
                    "MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"), F.upper("borough"))
                 .otherwise("UNSPECIFIED"))
    .withColumn("complaint_type", F.trim("complaint_type"))
    .withColumn("resolution_hours",
                F.when(F.col("closed_ts") >= F.col("created_ts"),
                       (F.col("closed_ts").cast("long") - F.col("created_ts").cast("long")) / 3600))
    .withColumn("created_date_only", F.to_date("created_ts"))
    .withColumn("created_hour", F.hour("created_ts"))
    .select("unique_key", "created_ts", "closed_ts", "created_date_only", "created_hour", "agency",
            "complaint_type", "descriptor", "borough", "status", "resolution_hours"))

c311_report = c311_raw.agg(
    F.count("*").alias("raw rows"),
    F.countDistinct("unique_key").alias("unique requests"),
    F.sum(F.when(F.col("closed_date").isNull(), 1).otherwise(0)).alias("not closed yet"),
    F.sum(F.when(~F.upper("borough").isin("MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND")
                 | F.col("borough").isNull(), 1).otherwise(0)).alias("borough unspecified"),
).toPandas().T.rename(columns={0: "rows"})
print(f"Clean 311 rows: {c311.count():,}")
c311_report

### 4d. 🚦 Live traffic: keep only sensors reporting a valid speed (`status = 0`, `speed > 0`)

In [ ]:
traffic = (traffic_raw
    .select(F.col("link_id"), F.col("link_name"), F.upper("borough").alias("borough"),
            F.expr("try_cast(speed AS DOUBLE)").alias("speed_mph"),
            F.expr("try_cast(travel_time AS INT)").alias("travel_time_s"),
            F.expr("try_cast(data_as_of AS TIMESTAMP)").alias("reading_ts"),
            F.col("status"))
    .filter((F.col("status") == "0") & (F.col("speed_mph") > 0)))

# A sensor link reports many times; keep only its newest reading (window function)
newest_first = Window.partitionBy("link_id").orderBy(F.desc("reading_ts"))
traffic_now = (traffic.withColumn("rn", F.row_number().over(newest_first))
                      .filter(F.col("rn") == 1).drop("rn"))
print(f"Valid readings: {traffic.count():,}   Road links with a current speed: {traffic_now.count():,}")
traffic_now.agg(F.min("reading_ts").alias("oldest reading"), F.max("reading_ts").alias("newest reading")).show()

> 📸 **SCREENSHOT 8** — the 311 cleaning report and the live-traffic counts above.

### 4e. Queries and aggregations

Every query below is a chain of transformations, run by one action (`toPandas()`).
The results are small, so we convert them to **Pandas** for display and **matplotlib** charts.

In [ ]:
def save_result(pdf, name):
    pdf.to_csv(f"{RESULTS}/{name}.csv", index=False)

def save_chart(fig, name):
    fig.tight_layout()
    fig.savefig(f"{CHARTS}/{name}.png", dpi=120)
    plt.show()

## 🚕 Mobility
#### Q1. When does the city move? Taxi trips, fare and distance by hour — `groupBy` + `agg`

In [ ]:
spark.sparkContext.setJobDescription("Q1: taxi demand by hour")
q1 = (trips.groupBy("pickup_hour")
           .agg(F.count("*").alias("trips"),
                F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
                F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"))
           .orderBy("pickup_hour"))
q1_pd = q1.toPandas()
save_result(q1_pd, "q1_taxi_demand_by_hour")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(q1_pd.pickup_hour, q1_pd.trips, color="#4C72B0")
ax.set(title="Q1. Taxi trips by hour of day (Jan 2024)", xlabel="Hour of day", ylabel="Trips")
ax.set_xticks(range(24))
save_chart(fig, "q1_taxi_demand_by_hour")
q1_pd

**The same query in Spark SQL.** DataFrame API and SQL compile to the same plan and give the same result.

In [ ]:
trips.createOrReplaceTempView("trips")
q1_sql = spark.sql("""
    SELECT pickup_hour,
           COUNT(*)                     AS trips,
           ROUND(AVG(fare_amount), 2)   AS avg_fare,
           ROUND(AVG(trip_distance), 2) AS avg_distance_mi
    FROM trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""")
print("SQL result identical to DataFrame result:", q1_sql.toPandas().equals(q1_pd))

#### Q2. Busiest mobility hubs — top 10 pickup zones (`join` with the zones CSV)

In [ ]:
spark.sparkContext.setJobDescription("Q2: busiest mobility hubs")
q2 = (trips.join(zones, trips.PULocationID == zones.LocationID)
           .groupBy("Zone", "Borough")
           .agg(F.count("*").alias("pickups"),
                F.round(F.sum("total_amount"), 0).alias("revenue_usd"))
           .orderBy(F.desc("pickups"))
           .limit(10))
q2_pd = q2.toPandas()
save_result(q2_pd, "q2_mobility_hubs")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(q2_pd.Zone[::-1], q2_pd.pickups[::-1] / 1e3, color="#55A868")
ax.set(title="Q2. Top 10 taxi pickup zones", xlabel="Pickups (thousands)")
save_chart(fig, "q2_mobility_hubs")
q2_pd

#### Q3. Weekly mobility pattern — `pivot` (day of week × hour)

In [ ]:
spark.sparkContext.setJobDescription("Q3: weekly heatmap (pivot)")
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
q3 = trips.groupBy("day_of_week").pivot("pickup_hour", list(range(24))).count()
q3_pd = q3.toPandas().set_index("day_of_week").reindex(DAYS).fillna(0)
save_result(q3_pd.reset_index(), "q3_weekly_heatmap")

fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(q3_pd.values, aspect="auto", cmap="YlOrRd")
ax.set(title="Q3. Taxi trips by day of week and hour", xlabel="Hour of day")
ax.set_xticks(range(24)); ax.set_yticks(range(7)); ax.set_yticklabels(DAYS)
fig.colorbar(im, ax=ax, label="Trips")
save_chart(fig, "q3_weekly_heatmap")

## 🏛️ Citizen services (311)
#### Q4. What do citizens complain about most?

In [ ]:
spark.sparkContext.setJobDescription("Q4: top 311 complaint types")
q4 = (c311.groupBy("complaint_type")
          .agg(F.count("*").alias("requests"))
          .withColumn("share_pct", F.round(F.col("requests") / F.sum("requests").over(Window.partitionBy()) * 100, 1))
          .orderBy(F.desc("requests"))
          .limit(10))
q4_pd = q4.toPandas()
save_result(q4_pd, "q4_top_complaints")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(q4_pd.complaint_type[::-1], q4_pd.requests[::-1], color="#8172B2")
ax.set(title="Q4. Top 10 311 complaint types (Jan 2024)", xlabel="Requests")
save_chart(fig, "q4_top_complaints")
q4_pd

#### Q5. How fast does each city agency resolve requests? — `percentile_approx` (median)

In [ ]:
spark.sparkContext.setJobDescription("Q5: agency resolution time")
q5 = (c311.filter(F.col("resolution_hours").isNotNull())
          .groupBy("agency")
          .agg(F.count("*").alias("closed_requests"),
               F.round(F.percentile_approx("resolution_hours", 0.5), 1).alias("median_hours"),
               F.round(F.avg("resolution_hours"), 1).alias("avg_hours"))
          .filter(F.col("closed_requests") >= 500)
          .orderBy("median_hours"))
q5_pd = q5.toPandas()
save_result(q5_pd, "q5_agency_resolution_time")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(q5_pd.agency, q5_pd.median_hours, color="#C44E52")
ax.set(title="Q5. Median hours to close a 311 request, by agency", ylabel="Hours (median)")
ax.set_yscale("log")
save_chart(fig, "q5_agency_resolution_time")
q5_pd

#### Q6. Top 3 complaint types in each borough — **window function** `dense_rank()`

In [ ]:
spark.sparkContext.setJobDescription("Q6: top complaints per borough (window)")
by_borough = Window.partitionBy("borough").orderBy(F.desc("requests"))
q6 = (c311.filter(F.col("borough") != "UNSPECIFIED")
          .groupBy("borough", "complaint_type").agg(F.count("*").alias("requests"))
          .withColumn("rank_in_borough", F.dense_rank().over(by_borough))
          .filter(F.col("rank_in_borough") <= 3)
          .orderBy("borough", "rank_in_borough"))
q6_pd = q6.toPandas()
save_result(q6_pd, "q6_top_complaints_per_borough")
q6_pd

## 🌫️🌦️ Environment
#### Q7. Does traffic affect air quality? Hourly taxi activity joined with NO₂ and PM2.5 — `join` + `corr`

In [ ]:
spark.sparkContext.setJobDescription("Q7: taxi activity vs air quality")
hourly_trips = (trips.groupBy(F.date_trunc("hour", "pickup_ts").alias("hour_ts"))
                     .agg(F.count("*").alias("taxi_trips")))
traffic_air = hourly_trips.join(air, "hour_ts").na.drop()

q7 = traffic_air.agg(F.round(F.corr("taxi_trips", "nitrogen_dioxide"), 3).alias("corr_trips_vs_NO2"),
                     F.round(F.corr("taxi_trips", "pm2_5"), 3).alias("corr_trips_vs_PM2_5"),
                     F.round(F.corr("taxi_trips", "carbon_monoxide"), 3).alias("corr_trips_vs_CO"))
q7_pd = q7.toPandas()
save_result(q7_pd, "q7_traffic_vs_air_quality")

by_hour = (traffic_air.groupBy(F.hour("hour_ts").alias("hour"))
                      .agg(F.avg("taxi_trips").alias("taxi_trips"), F.avg("nitrogen_dioxide").alias("no2"))
                      .orderBy("hour").toPandas())
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(by_hour.hour, by_hour.taxi_trips, color="#4C72B0", alpha=0.6, label="Taxi trips")
ax1.set(xlabel="Hour of day", ylabel="Avg taxi trips per hour")
ax2 = ax1.twinx()
ax2.plot(by_hour.hour, by_hour.no2, "o-", color="#C44E52", label="NO₂")
ax2.set_ylabel("Avg NO₂ (µg/m³)")
ax1.set_title("Q7. Average taxi activity and NO₂ by hour of day")
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.9))
save_chart(fig, "q7_traffic_vs_air_quality")
q7_pd

#### Q8. Do cold days bring more heating complaints? Daily 311 *Heat/Hot Water* requests vs temperature

In [ ]:
spark.sparkContext.setJobDescription("Q8: temperature vs heating complaints")
daily_heat = (c311.filter(F.upper("complaint_type") == "HEAT/HOT WATER")
                  .groupBy(F.col("created_date_only").alias("date"))
                  .agg(F.count("*").alias("heat_complaints")))
daily_temp = (weather.groupBy(F.to_date("hour_ts").alias("date"))
                     .agg(F.round(F.avg("temp_c"), 1).alias("avg_temp_c"),
                          F.round(F.sum("precip_mm"), 1).alias("precip_mm")))
q8 = daily_heat.join(daily_temp, "date").orderBy("date")
q8_pd = q8.toPandas()
heat_corr = q8.agg(F.corr("heat_complaints", "avg_temp_c")).first()[0]
save_result(q8_pd, "q8_temperature_vs_heat_complaints")
print(f"Correlation between daily temperature and heating complaints: {heat_corr:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(q8_pd.avg_temp_c, q8_pd.heat_complaints, color="#DD8452")
ax.set(title=f"Q8. Daily heating complaints vs temperature (r = {heat_corr:.2f})",
       xlabel="Average daily temperature (°C)", ylabel="Heat/Hot Water complaints")
save_chart(fig, "q8_temperature_vs_heat_complaints")
q8_pd.head(10)

## 🚦 Live traffic
#### Q9. How fast is traffic moving right now? Live speed by borough and the slowest roads

In [ ]:
spark.sparkContext.setJobDescription("Q9: live traffic speeds")
q9 = (traffic_now.groupBy("borough")
                 .agg(F.count("*").alias("road_links"),
                      F.round(F.avg("speed_mph"), 1).alias("avg_speed_mph"),
                      F.round(F.min("speed_mph"), 1).alias("min_speed_mph"))
                 .orderBy("avg_speed_mph"))
q9_pd = q9.toPandas()
save_result(q9_pd, "q9_live_speed_by_borough")

slowest = (traffic_now.orderBy("speed_mph")
                      .select("borough", "link_name", "speed_mph", "reading_ts").limit(10).toPandas())
save_result(slowest, "q9_slowest_roads_now")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(q9_pd.borough, q9_pd.avg_speed_mph, color="#64B5CD")
ax.set(title=f"Q9. Live average road speed by borough (snapshot {slowest.reading_ts.max()})",
       ylabel="mph")
save_chart(fig, "q9_live_speed_by_borough")
display(q9_pd)
slowest

## 🏙️ Integrated view
#### Q10. Borough smart-city scorecard — one table built from **four** sources

Mobility (taxi + zones), citizen services (311), live traffic, and borough population
(a small DataFrame created directly from Python with `createDataFrame`). Boroughs are matched on a normalized
upper-case name, with outer joins so a borough missing from one source still appears.

In [ ]:
spark.sparkContext.setJobDescription("Q10: borough scorecard")
population = spark.createDataFrame(
    [("BRONX", 1472654), ("BROOKLYN", 2736074), ("MANHATTAN", 1694251),
     ("QUEENS", 2405464), ("STATEN ISLAND", 495747)],              # 2020 US Census
    ["borough", "population"])

mobility = (trips.join(F.broadcast(zones), trips.PULocationID == zones.LocationID)
                 .groupBy(F.upper("Borough").alias("borough"))
                 .agg(F.count("*").alias("taxi_pickups")))
services = (c311.groupBy("borough")
                .agg(F.count("*").alias("requests_311"),
                     F.round(F.percentile_approx("resolution_hours", 0.5), 1).alias("median_resolution_h")))
live = traffic_now.groupBy("borough").agg(F.round(F.avg("speed_mph"), 1).alias("live_speed_mph"))

q10 = (population
       .join(mobility, "borough", "left")
       .join(services, "borough", "left")
       .join(live, "borough", "left")
       .withColumn("pickups_per_1k_people", F.round(F.col("taxi_pickups") / F.col("population") * 1000, 1))
       .withColumn("requests_per_1k_people", F.round(F.col("requests_311") / F.col("population") * 1000, 1))
       .orderBy(F.desc("population")))
q10_pd = q10.toPandas()
save_result(q10_pd, "q10_borough_scorecard")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col, title, color in zip(axes,
        ["pickups_per_1k_people", "requests_per_1k_people", "live_speed_mph"],
        ["Taxi pickups per 1k people", "311 requests per 1k people", "Live road speed (mph)"],
        ["#4C72B0", "#8172B2", "#64B5CD"]):
    ax.bar(q10_pd.borough.str.title(), q10_pd[col].fillna(0), color=color)
    ax.set_title(title); ax.tick_params(axis="x", rotation=30)
fig.suptitle("Q10. Borough smart-city scorecard")
save_chart(fig, "q10_borough_scorecard")
q10_pd

> 📸 **SCREENSHOT 9** — the query outputs and charts Q1–Q10 above.

`explain()` shows how Spark will execute a query. Read it bottom-up: scan → filter → partial aggregate
→ **Exchange (shuffle)** → final aggregate.

In [ ]:
q1.explain()
spark.sparkContext.setJobDescription(None)

## Step 5 — Optimize performance (cache, partitions, shuffle)

Each experiment runs a **baseline** and an **optimized** version of the same work. We report the **median of 3 runs**.
To time the full computation without the cost of collecting or saving output, we use Spark's `noop` writer: it runs
the whole plan and then discards the rows.

In [ ]:
RUNS = 3
opt_results = []

def run_query(df):
    """Execute the full plan of df and discard the output."""
    df.write.format("noop").mode("overwrite").save()

def median_time(fn, runs=RUNS):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)

def record(name, baseline_desc, baseline_s, optimized_desc, optimized_s):
    opt_results.append({"optimization": name,
                        "baseline": baseline_desc, "baseline_s": round(baseline_s, 2),
                        "optimized": optimized_desc, "optimized_s": round(optimized_s, 2),
                        "speedup_x": round(baseline_s / optimized_s, 2)})
    print(f"{name}: {baseline_desc} = {baseline_s:.2f}s  ->  {optimized_desc} = {optimized_s:.2f}s  "
          f"(speed-up {baseline_s / optimized_s:.2f}x)")

### O1. Caching

Without a cache, **every** query re-reads the Parquet file and re-applies all cleaning steps.
`cache()` keeps the cleaned DataFrame in memory after the first action, so later queries start from there.

In [ ]:
def three_queries(df):
    run_query(df.groupBy("pickup_hour").count())
    run_query(df.groupBy("payment_method").agg(F.avg("tip_pct")))
    run_query(df.groupBy("PULocationID").agg(F.sum("total_amount")))

spark.sparkContext.setJobDescription("O1 baseline: no cache")
no_cache_s = median_time(lambda: three_queries(trips))

spark.sparkContext.setJobDescription("O1: build cache")
trips.cache()
t0 = time.perf_counter(); trips.count(); cache_build_s = time.perf_counter() - t0
print(f"One-time cost to build the cache: {cache_build_s:.2f}s")

spark.sparkContext.setJobDescription("O1 optimized: cached")
cached_s = median_time(lambda: three_queries(trips))
record("O1 Caching", "3 queries, no cache", no_cache_s, "3 queries, cached", cached_s)

> 📸 **SCREENSHOT 10** — Spark UI → **Storage** tab (shows the cached DataFrame, its size in memory, and 100% cached).

### O2. Broadcast join vs sort-merge join

Joining 3M trips with 265 zones: a **sort-merge join** shuffles and sorts *both* sides.
A **broadcast join** copies the small table to every task, so the big table is never shuffled.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # turn off automatic broadcast for the baseline

def revenue_by_borough(zone_df):
    return (trips.join(zone_df, trips.PULocationID == zone_df.LocationID)
                 .groupBy("Borough").agg(F.sum("total_amount")))

smj = revenue_by_borough(zones)
bhj = revenue_by_borough(F.broadcast(zones))                     # broadcast hint

print("Baseline plan  :", "SortMergeJoin" if "SortMergeJoin" in smj._jdf.queryExecution().toString() else "?")
print("Optimized plan :", "BroadcastHashJoin" if "BroadcastHashJoin" in bhj._jdf.queryExecution().toString() else "?")
bhj.explain()

spark.sparkContext.setJobDescription("O2 baseline: sort-merge join")
smj_s = median_time(lambda: run_query(smj))
spark.sparkContext.setJobDescription("O2 optimized: broadcast join")
bhj_s = median_time(lambda: run_query(bhj))
record("O2 Broadcast join", "sort-merge join", smj_s, "broadcast hash join", bhj_s)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # restore the default (10 MB)

> 📸 **SCREENSHOT 11** — Spark UI → **SQL / DataFrame** tab → open the 'O2 optimized: broadcast join' query (diagram shows *BroadcastHashJoin*), and the 'O2 baseline' one (shows *SortMergeJoin* + 2 Exchanges).

### O3. Shuffle partitions and Adaptive Query Execution (AQE)

After a shuffle, Spark uses `spark.sql.shuffle.partitions` partitions (**default 200**). For a few million rows on a
few cores, 200 tiny tasks mostly add scheduling overhead. We switch AQE off to see the raw effect, then show
that AQE fixes the default automatically by merging small partitions.

In [ ]:
def zone_pairs():
    return (trips.groupBy("PULocationID", "DOLocationID")
                 .agg(F.count("*").alias("trips"), F.avg("fare_amount").alias("avg_fare")))

spark.conf.set("spark.sql.adaptive.enabled", "false")

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.sparkContext.setJobDescription("O3 baseline: 200 shuffle partitions, AQE off")
p200_s = median_time(lambda: run_query(zone_pairs()))

tuned = str(spark.sparkContext.defaultParallelism * 2)
spark.conf.set("spark.sql.shuffle.partitions", tuned)
spark.sparkContext.setJobDescription(f"O3 optimized: {tuned} shuffle partitions, AQE off")
tuned_s = median_time(lambda: run_query(zone_pairs()))
record("O3 Shuffle partitions", "200 partitions (default)", p200_s, f"{tuned} partitions (2 x cores)", tuned_s)

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.sparkContext.setJobDescription("O4 optimized: 200 partitions, AQE on")
aqe_s = median_time(lambda: run_query(zone_pairs()))
record("O4 AQE", "200 partitions, AQE off", p200_s, "200 partitions, AQE on", aqe_s)

### O5. File format: CSV vs Parquet

We save the same raw trips as CSV once, then run the same query on both files.
Parquet is **columnar** and compressed: Spark reads only the 2 columns the query needs (*column pruning*)
and skips data using min/max statistics (*predicate pushdown*). CSV must be read and parsed in full.

In [ ]:
CSV_PATH = f"{PROCESSED}/trips_csv"
spark.sparkContext.setJobDescription("O5: write CSV copy")
if not os.path.exists(CSV_PATH):
    trips_raw.write.mode("overwrite").option("header", True).csv(CSV_PATH)

def avg_fare_by_payment(df):
    return df.filter(F.col("fare_amount") > 0).groupBy("payment_type").agg(F.avg("fare_amount"))

csv_df = spark.read.option("header", True).schema(trips_raw.schema).csv(CSV_PATH)
parquet_df = spark.read.parquet(TRIPS_PATH)

spark.sparkContext.setJobDescription("O5 baseline: CSV")
csv_s = median_time(lambda: run_query(avg_fare_by_payment(csv_df)))
spark.sparkContext.setJobDescription("O5 optimized: Parquet")
parquet_s = median_time(lambda: run_query(avg_fare_by_payment(parquet_df)))
record("O5 File format", "CSV", csv_s, "Parquet", parquet_s)

print(f"\nStorage size: CSV {size_mb(CSV_PATH):.0f} MB  vs  Parquet {size_mb(TRIPS_PATH):.0f} MB")
print("\nParquet scan reads only these columns (column pruning) and pushes the filter down:")
plan = avg_fare_by_payment(parquet_df)._jdf.queryExecution().executedPlan().toString()
print("\n".join(line.strip() for line in plan.splitlines() if "FileScan" in line))
spark.sparkContext.setJobDescription(None)

### Optimization summary

In [ ]:
opt_df = pd.DataFrame(opt_results)
save_result(opt_df, "optimization_results")
opt_df

> 📸 **SCREENSHOT 12** — the optimization summary table above.

## Step 6 — Capture results

We save the cleaned dataset as Parquet (the format downstream jobs should read), and list all saved outputs:
query results as CSV and charts as PNG.

In [ ]:
spark.sparkContext.setJobDescription("Step 6: write cleaned trips")
CLEAN_PATH = f"{PROCESSED}/trips_clean_parquet"
trips.write.mode("overwrite").parquet(CLEAN_PATH)
print(f"Cleaned trips saved to {CLEAN_PATH} ({size_mb(CLEAN_PATH):.0f} MB, "
      f"{spark.read.parquet(CLEAN_PATH).count():,} rows)")

for folder in (RESULTS, CHARTS):
    print(f"\n{folder}/")
    for name in sorted(os.listdir(folder)):
        print("   ", name)
spark.sparkContext.setJobDescription(None)

## Step 7 — Performance analysis

### 7a. Effect of each optimization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
y = range(len(opt_df))
ax.barh([i + 0.2 for i in y], opt_df.baseline_s, height=0.4, label="Baseline", color="#C44E52")
ax.barh([i - 0.2 for i in y], opt_df.optimized_s, height=0.4, label="Optimized", color="#55A868")
ax.set_yticks(list(y)); ax.set_yticklabels(opt_df.optimization); ax.invert_yaxis()
for i, row in opt_df.iterrows():
    ax.text(max(row.baseline_s, row.optimized_s) * 1.02, i, f"{row.speedup_x}x", va="center")
ax.set(title="Baseline vs optimized (median of 3 runs)", xlabel="Seconds (lower is better)")
ax.set_xlim(0, opt_df[["baseline_s", "optimized_s"]].max().max() * 1.15)
ax.legend()
save_chart(fig, "7a_optimization_speedup")

### 7b. Pandas vs Spark on growing data sizes

Same job in both tools: read the file, filter, compute trips and average fare per hour.
Pandas runs in one process on one core and must hold all the data in memory; Spark splits the work into parallel tasks.

In [ ]:
total_rows = trips_raw.count()
sizes = [n for n in (100_000, 500_000, 1_000_000) if n < total_rows] + [total_rows]

spark.sparkContext.setJobDescription("7b: write samples")
sample_paths = {}
for n in sizes:
    path = TRIPS_PATH if n == total_rows else f"{PROCESSED}/sample_{n}"
    if n != total_rows and not os.path.exists(path):
        trips_raw.limit(n).write.mode("overwrite").parquet(path)
    sample_paths[n] = path

def pandas_job(path):
    pdf = pd.read_parquet(path, columns=["tpep_pickup_datetime", "fare_amount"])
    pdf = pdf[pdf.fare_amount > 0]
    return pdf.groupby(pdf.tpep_pickup_datetime.dt.hour)["fare_amount"].agg(["count", "mean"])

def spark_job(path):
    df = spark.read.parquet(path).filter(F.col("fare_amount") > 0)
    return (df.groupBy(F.hour(F.col("tpep_pickup_datetime").cast("timestamp")).alias("hour"))
              .agg(F.count("*"), F.avg("fare_amount")).collect())

spark.sparkContext.setJobDescription("7b: Spark vs Pandas")
comparison = []
for n, path in sample_paths.items():
    comparison.append({"rows": n,
                       "pandas_s": round(median_time(lambda: pandas_job(path)), 3),
                       "spark_s":  round(median_time(lambda: spark_job(path)), 3)})
cmp_df = pd.DataFrame(comparison)
save_result(cmp_df, "pandas_vs_spark")

full_pdf_mb = pd.read_parquet(TRIPS_PATH).memory_usage(deep=True).sum() / 1e6
print(f"Pandas needs {full_pdf_mb:,.0f} MB of RAM to hold the full month in memory "
      f"(the file on disk is {size_mb(TRIPS_PATH):.0f} MB).")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cmp_df.rows / 1e6, cmp_df.pandas_s, "o-", label="Pandas", color="#DD8452")
ax.plot(cmp_df.rows / 1e6, cmp_df.spark_s, "o-", label="Spark", color="#4C72B0")
ax.set(title="Pandas vs Spark: same job, growing data", xlabel="Rows (millions)", ylabel="Seconds")
ax.legend()
save_chart(fig, "7b_pandas_vs_spark")
spark.sparkContext.setJobDescription(None)
cmp_df

> 📸 **SCREENSHOT 13** — the Pandas vs Spark table and chart.

### 7c. Scaling with CPU cores (comparing execution modes)

The same pipeline (read → clean → join → aggregate) runs with 1 core (`local[1]`), 2 cores and all cores (`local[*]`).
This restarts Spark for each setting.

> ⚠️ **Take all the Spark UI screenshots (4, 5, 6, 10, 11) BEFORE running the next cell.** Restarting Spark clears the UI.

In [ ]:
def pipeline_workload(session):
    raw = session.read.parquet(TRIPS_PATH)
    zone_df = session.read.csv(ZONES_PATH, header=True, schema=zone_schema)
    cleaned = (raw.withColumn("pickup_ts", F.col("tpep_pickup_datetime").cast("timestamp"))
                  .filter((F.col("fare_amount") > 0) & (F.col("trip_distance") > 0)))
    result = (cleaned.join(F.broadcast(zone_df), cleaned.PULocationID == zone_df.LocationID)
                     .groupBy("Borough", F.hour("pickup_ts").alias("hour"))
                     .agg(F.count("*"), F.avg("fare_amount"), F.sum("total_amount")))
    result.write.format("noop").mode("overwrite").save()

cpu_count = os.cpu_count() or 1
modes = ["local[1]"] + (["local[2]"] if cpu_count > 2 else []) + ["local[*]"]

scaling = []
for mode in modes:
    spark.stop()
    spark = create_spark(mode)
    spark.sparkContext.setLogLevel("ERROR")
    cores = spark.sparkContext.defaultParallelism
    seconds = median_time(lambda: pipeline_workload(spark))
    scaling.append({"mode": mode, "cores": cores, "seconds": round(seconds, 2)})
    print(f"{mode:9s} ({cores} cores): {seconds:.2f}s")

scale_df = pd.DataFrame(scaling)
scale_df["speedup_vs_1_core"] = (scale_df.seconds.iloc[0] / scale_df.seconds).round(2)
save_result(scale_df, "scaling_by_cores")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([f"{m}\n{c} cores" for m, c in zip(scale_df["mode"], scale_df.cores)], scale_df.seconds, color="#4C72B0")
ax.set(title="Same pipeline, different number of cores", ylabel="Seconds (lower is better)")
save_chart(fig, "7c_scaling_by_cores")
scale_df

> 📸 **SCREENSHOT 14** — the scaling table and chart.

### 7d. Smart city insights (computed from the query results)

In [ ]:
peak = q1_pd.loc[q1_pd.trips.idxmax()]
top_hub = q2_pd.iloc[0]
top_complaint = q4_pd.iloc[0]
fastest_agency, slowest_agency = q5_pd.iloc[0], q5_pd.iloc[-1]
q7_row = q7_pd.iloc[0]

print(f"🚕 Peak taxi hour: {int(peak.pickup_hour)}:00 with {int(peak.trips):,} trips in the month; "
      f"busiest hub: {top_hub.Zone} ({int(top_hub.pickups):,} pickups).")
print(f"🏛️ Most common 311 complaint: {top_complaint.complaint_type} ({top_complaint.share_pct}% of requests).")
print(f"🏛️ Fastest agency: {fastest_agency.agency} (median {fastest_agency.median_hours} h); "
      f"slowest: {slowest_agency.agency} (median {slowest_agency.median_hours} h).")
print(f"🌫️ Correlation of hourly taxi trips with NO₂: {q7_row.corr_trips_vs_NO2}, with PM2.5: {q7_row.corr_trips_vs_PM2_5}.")
print(f"🌦️ Correlation of daily temperature with heating complaints: {heat_corr:.2f} "
      f"({'colder days bring more complaints' if heat_corr < 0 else 'no clear cold-weather effect'}).")
if len(q9_pd):
    print(f"🚦 Live: slowest borough right now is {q9_pd.iloc[0].borough.title()} "
          f"({q9_pd.iloc[0].avg_speed_mph} mph); slowest road: {slowest.iloc[0].link_name} ({slowest.iloc[0].speed_mph} mph).")

### 7e. Summary of measured results

In [ ]:
print("OPTIMIZATIONS")
for r in opt_results:
    print(f"  {r['optimization']:24s} {r['baseline_s']:7.2f}s -> {r['optimized_s']:7.2f}s   {r['speedup_x']}x faster")

print("\nPANDAS vs SPARK")
for r in comparison:
    winner = "Pandas" if r["pandas_s"] < r["spark_s"] else "Spark"
    print(f"  {r['rows']:>10,} rows: Pandas {r['pandas_s']:6.2f}s | Spark {r['spark_s']:6.2f}s  -> {winner} faster")

print("\nSCALING BY CORES")
for r in scale_df.to_dict("records"):
    print(f"  {r['mode']:9s} {r['cores']} cores: {r['seconds']:6.2f}s  ({r['speedup_vs_1_core']}x vs 1 core)")

### Conclusions

**Performance**

1. **Caching** gives the biggest win when the same cleaned data feeds several queries. Without it, Spark re-reads and
   re-cleans the source for every action, because DataFrames are lazy and do not keep results.
2. **Broadcast joins** remove the shuffle of the large table when one side is small (265 zones). Spark broadcasts tables
   under 10 MB automatically; the hint makes it explicit.
3. **Shuffle partitions:** the default of 200 is tuned for big clusters. For ~3M rows on a few cores, fewer partitions
   cut task overhead. **AQE** gets most of this benefit automatically by merging small partitions at runtime.
4. **Parquet beats CSV** on both speed and storage: it is compressed, columnar (reads only the needed columns) and
   skips data with statistics. CSV must be parsed text, every column, every time.
5. **Pandas vs Spark:** on small data Pandas is faster, because Spark pays a fixed cost to plan and schedule tasks.
   As data grows, Spark's parallelism closes the gap. Pandas must fit everything in one machine's RAM; Spark can
   spill to disk and scale out to a cluster with the same code.
6. **More cores → faster**, until the job is limited by I/O or the remaining serial work (Amdahl's law). On a real
   cluster the same code scales across machines by changing only the `master` setting.

**Smart city value**

7. One engine handled six sources in four formats (Parquet, CSV, JSON, live API) with the same DataFrame API,
   and joined them by time (hourly/daily) and place (borough) to answer cross-domain questions.
8. The same pipeline can run on a schedule: re-running it pulls a new live-traffic snapshot, which is the first
   step towards a real-time city dashboard (Spark Structured Streaming would make it continuous).

In [ ]:
# Zip all charts and result tables (downloads automatically in Colab)
archive = shutil.make_archive("case_study_outputs", "zip", "outputs")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Saved", archive)